# Split data into train, valid, and test (?)

Logan Wong

law3082

In [1]:
import os
import pandas as pd
import numpy as np
import torch

In [2]:
# Load the data generated by preprocess.py
df = pd.read_csv('./twitter-incremental-clustering/event2012.tsv', sep='\t')

total_num_tweets = len(df)
print(f"Loaded {total_num_tweets} tweets.")

n = df['label'].nunique()
print(f"Unique Events (Labels): {n}")

Loaded 68841 tweets.
Unique Events (Labels): 503


In [8]:
embeddings_raw = np.load('data/event2012_embeddings.npy')
# Convert to a Float Tensor & move to GPU
embeddings = torch.from_numpy(embeddings_raw).float()

# Load the metadata to track tweet IDs
metadata = pd.read_csv('data/event2012_metadata.csv')

print(f"Loaded {embeddings.shape[0]} embeddings with dimension {embeddings.shape[1]}")

Loaded 68841 embeddings with dimension 768


In [4]:
# Convert to datetime
df['created_at'] = pd.to_datetime(df['created_at'])

# Sort chronologically
df = df.sort_values('created_at').reset_index(drop=True)

# Define Cutoffs
# Week 1-2 (Days 0-14)
train_cutoff = df['created_at'].min() + pd.Timedelta(days=14)
# Week 3 (Days 14-21)
valid_cutoff = df['created_at'].min() + pd.Timedelta(days=21)

# Create Masks
train_mask = df['created_at'] < train_cutoff
valid_mask = (df['created_at'] >= train_cutoff) & (df['created_at'] < valid_cutoff)
test_mask = df['created_at'] >= valid_cutoff

# Apply Masks to Embeddings
train_embeddings = embeddings[train_mask]
valid_embeddings = embeddings[valid_mask]
test_embeddings = embeddings[test_mask]

# Apply Masks to DataFrame
train_df = df[train_mask]
valid_df = df[valid_mask]
test_df = df[test_mask]

print("Split data successfully!")

Split data successfully!


In [5]:
rows_in_train = train_mask.sum()
rows_in_val = valid_mask.sum()
rows_in_test = test_mask.sum()

print(f"Rows in Train: {rows_in_train}")
print(f"Rows in Valid: {rows_in_val}")
print(f"Rows in Test:  {rows_in_test}")

sanity_check = rows_in_train + rows_in_val + rows_in_test == total_num_tweets
print(f"They add up correctly: {sanity_check}")

Rows in Train: 42700
Rows in Valid: 13417
Rows in Test:  12724
They add up correctly: True


In [9]:
# Save the 3 splits
np.save('data/train_embeddings.npy', train_embeddings)
np.save('data/valid_embeddings.npy', valid_embeddings)
np.save('data/test_embeddings.npy', test_embeddings)

train_df[['id', 'label']].to_csv('data/train_metadata.csv', index=False)
valid_df[['id', 'label']].to_csv('data/valid_metadata.csv', index=False)
test_df[['id', 'label']].to_csv('data/test_metadata.csv', index=False)

print(f"Train: {len(train_df)}")
print(f"Valid: {len(valid_df)}")
print(f"Test: {len(test_df)}")

Train: 42700
Valid: 13417
Test: 12724
